# LLM Jury - Quickstart Tutorial

**Goal:** Run your first evaluation in under 2 minutes!

In this tutorial, you'll learn how to:
1. Set up a single LLM judge
2. Create an evaluation metric
3. Run an evaluation
4. Interpret the results

---

## Step 1: Installation

First, make sure you have the library installed:

```bash
pip install git+https://github.com/netraj-patil/LLM-Jury.git
pip install langchain-groq  # or langchain-openai, langchain-google-genai, etc.
```

## Step 2: API Key Setup

You'll need an API key from one of these providers:
- [Groq](https://console.groq.com/) (Free tier available)
- [OpenAI](https://platform.openai.com/)
- [Google AI](https://ai.google.dev/)
- [Anthropic](https://console.anthropic.com/)

For this tutorial, we'll use **Groq** (fast and free!).

In [ ]:
# Set your API key here
import os

# Option 1: Set directly (not recommended for production)
GROQ_API_KEY = "gsk_YOUR_API_KEY_HERE"

# Option 2: Use environment variable (recommended)
# os.environ["GROQ_API_KEY"] = "your-key-here"

# Option 3: Load from .env file
# from dotenv import load_dotenv
# load_dotenv()

## Step 3: Import Required Components

Let's import everything we need from the LLM Jury library.

In [2]:
# LangChain model (we'll use Groq for this example)
from langchain_groq import ChatGroq

# LLM Jury components
from llm_jury.core.evaluator import JuryEvaluator
from llm_jury.judges.llm_judge import LLMJudge
from llm_jury.metrics.predefined import GroundednessMetric
from llm_jury.strategies.consensus import MajorityVoting

print("✅ All imports successful!")

✅ All imports successful!


## Step 4: Initialize the Judge

A **Judge** is an LLM that evaluates text based on specific criteria. Let's create one using Groq's Llama model.

In [4]:
# Initialize the LLM model
llm_model = ChatGroq(
    model_name="openai/gpt-oss-120b",
    temperature=0.0,  # Use 0 for deterministic evaluations
    api_key=GROQ_API_KEY
)

# Wrap it in an LLMJudge
judge = LLMJudge(
    model=llm_model,
    name="Openai-gpt-oss-120b"  # Give it a friendly name
)

print(f"Judge created: {judge.name}")

Judge created: Openai-gpt-oss-120b


## Step 5: Create an Evaluation Metric

A **Metric** defines *what* you're evaluating. We'll use the built-in **GroundednessMetric**, which checks if the output is supported by the source text (critical for RAG applications!).

In [5]:
# Create the Groundedness metric
metric = GroundednessMetric()

print(f"✅ Metric created: {metric.name}")
print(f"   Description: {metric.description}")
print(f"   Scale: {metric.scale_min} to {metric.scale_max}")

✅ Metric created: Groundedness
   Description: Measures if the answer is derived solely from the source context.
   Scale: 1.0 to 5.0


## Step 6: Create the Jury Evaluator

The **JuryEvaluator** orchestrates the evaluation process. Even with one judge, it provides a consistent interface.

In [6]:
# Create the jury with our single judge
jury = JuryEvaluator(
    judges=[judge],  # List of judges (just one for now)
    strategy=MajorityVoting()  # Aggregation strategy (not needed for single judge, but good practice)
)

print(f"✅ Jury created with {len(jury.judges)} judge(s)")

✅ Jury created with 1 judge(s)


## Step 7: Prepare Your Evaluation Data

Let's create a simple example where we have:
- **Source Text**: The ground truth or retrieved context
- **Output Text**: The LLM's generated response

We'll test if the output is grounded in the source.

In [7]:
# Define the source context (ground truth)
source_text = """
The Apollo 11 mission successfully landed on the Moon on July 20, 1969. 
Neil Armstrong was the first person to walk on the lunar surface, followed by Buzz Aldrin. 
Michael Collins remained in orbit aboard the Command Module.
"""

# Define the model's output (what we're evaluating)
output_text = """
Neil Armstrong became the first human to walk on the Moon in 1969 during the Apollo 11 mission.
"""

# Package them into a context dictionary
context = {
    "source_text": source_text,
    "output_text": output_text
}

print("✅ Evaluation data prepared!")

✅ Evaluation data prepared!


## Step 8: Run the Evaluation!

This is where the magic happens. The jury will:
1. Extract features from the text
2. Generate a prompt based on the metric
3. Ask the judge to evaluate
4. Parse and normalize the score
5. Return a comprehensive result

In [8]:
# Run the evaluation
result = jury.evaluate(
    context=context,
    output=output_text,
    metric=metric
)

print("✅ Evaluation complete!")

✅ Evaluation complete!


## Step 9: Examine the Results

The `EvaluationResult` contains:
- **final_score**: The aggregated score from all judges
- **is_valid**: Whether the output passed the threshold
- **confidence**: How confident the jury is in the decision
- **manifest**: Detailed audit trail with individual scores and features

In [9]:
# Print the main results
print("="*60)
print("EVALUATION RESULTS")
print("="*60)
print(f"\nFinal Score: {result.final_score}/5")
print(f"Valid: {result.is_valid}")
print(f"Confidence: {result.confidence:.2%}")
print(f"\nRecommendation: {result.get_recommendation()}")
print("="*60)

EVALUATION RESULTS

Final Score: 5.0/5
Valid: True
Confidence: 100.00%

Recommendation: APPROVE: The content meets quality standards with high confidence.


## Step 10: Deep Dive into Judge Reasoning

Let's see *why* the judge gave this score.

In [10]:
# Access individual judge scores from the manifest
print("\nJUDGE FEEDBACK")
print("="*60)

for score in result.manifest.individual_scores:
    print(f"\nJudge: {score.judge_id}")
    print(f"Score: {score.score}")
    print(f"\nReasoning:\n{score.reasoning}")
    print("-"*60)


JUDGE FEEDBACK

Judge: Openai-gpt-oss-120b
Score: 5.0

Reasoning:
The output states that Neil Armstrong was the first human to walk on the Moon in 1969 during the Apollo 11 mission. All elements of this claim are directly supported by the source text: the source confirms Armstrong was the first person (human) to walk on the lunar surface, that this occurred during the Apollo 11 mission, and that the mission took place in 1969. No additional or unsupported information is introduced.
------------------------------------------------------------


## Step 11: Explore Extracted Features

The library automatically extracts linguistic features from your text!

In [11]:
# Access the extracted features
features = result.manifest.features

print("\n📈 EXTRACTED FEATURES")
print("="*60)
print(f"\n📝 Text Metrics:")
print(f"   Word Count: {features.get('word_count', 'N/A')}")
print(f"   Character Count: {features.get('char_count', 'N/A')}")
print(f"   Sentence Count: {features.get('sentence_count', 'N/A')}")
print(f"   Compression Ratio: {features.get('compression_ratio', 'N/A')}")

print(f"\n🎓 Complexity Metrics:")
print(f"   Flesch Reading Ease: {features.get('flesch_reading_ease', 'N/A')}")
print(f"   Lexical Diversity: {features.get('lexical_diversity', 'N/A')}")
print(f"   Avg Sentence Length: {features.get('avg_sentence_length', 'N/A')}")

print(f"\n🔬 Advanced Metrics:")
print(f"   Difficult Words: {features.get('difficult_word_count', 'N/A')}")
print(f"   Modality Verbs: {features.get('modality_verb_count', 'N/A')}")
print(f"   Shannon Entropy: {features.get('shannon_entropy', 'N/A')}")
print("="*60)


📈 EXTRACTED FEATURES

📝 Text Metrics:
   Word Count: 18
   Character Count: 97
   Sentence Count: 1
   Compression Ratio: 0.9485

🎓 Complexity Metrics:
   Flesch Reading Ease: 71.07
   Lexical Diversity: 0.8889
   Avg Sentence Length: 18.0

🔬 Advanced Metrics:
   Difficult Words: 1
   Modality Verbs: 0
   Shannon Entropy: 3.9058


## Step 12: Try a Negative Example

Let's test with an output that contains hallucinations (information NOT in the source).

In [12]:
# Create a hallucinated output
hallucinated_output = """
Neil Armstrong walked on Mars in 1969 and planted a flag with his crew of 5 astronauts.
"""

# Create new context
bad_context = {
    "source_text": source_text,
    "output_text": hallucinated_output
}

# Evaluate
bad_result = jury.evaluate(
    context=bad_context,
    output=hallucinated_output,
    metric=metric
)

# Show results
print("\n🚨 HALLUCINATED OUTPUT RESULTS")
print("="*60)
print(f"Final Score: {bad_result.final_score}/5")
print(f"Valid: {bad_result.is_valid}")
print(f"\nJudge Reasoning:")
print(bad_result.manifest.individual_scores[0].reasoning)
print("="*60)


🚨 HALLUCINATED OUTPUT RESULTS
Final Score: 1.0/5
Valid: False

Judge Reasoning:
The model output states that Neil Armstrong walked on Mars in 1969 and planted a flag with a crew of five astronauts. The source text only mentions that Armstrong walked on the Moon on July 20 1969, with Buzz Aldrin as the second astronaut, and Michael Collins remaining in orbit. There is no mention of Mars, a flag, or a crew of five, making the output contradictory and unrelated to the source.


## 🎉 Congratulations!

You've successfully:
- ✅ Set up an LLM judge
- ✅ Created an evaluation metric
- ✅ Run evaluations on good and bad outputs
- ✅ Interpreted the results and judge reasoning
- ✅ Explored automatic feature extraction

---

## 🚀 Next Steps

Now that you know the basics, explore:

1. **Multiple Judges**: Add more judges for consensus-based evaluation
2. **Custom Metrics**: Create your own evaluation criteria
3. **Different Strategies**: Try weighted voting or consensus thresholds
4. **Batch Evaluation**: Process multiple outputs at once
5. **Hallucination Shield**: Protect agentic workflows from errors

Check out the other tutorials:
- `02_The_Power_of_Consensus.ipynb`
- `03_Batch_Processing_Datasets.ipynb`
- `04_Aggregation_Strategies.ipynb`
- `05_Agentic_Shield.ipynb`

## 💡 Pro Tips

1. **Temperature**: Use `temperature=0.0` for consistent evaluations
2. **Multiple Judges**: 3-5 judges provide good consensus without excessive cost
3. **Metrics**: Combine multiple metrics for comprehensive evaluation
4. **Features**: Use extracted features for analysis and debugging
5. **Caching**: Store results for offline analysis and comparison

---

- [GitHub Repository](https://github.com/netraj-patil/LLM-Jury)

---

**Happy Evaluating! 🎯**